# CIC6314 — Smart Product Recommendation System
**Member 4 — Integration Layer**  
Dataset: UCI Online Retail | Pipeline: Rules Engine → BFS Search → RF Blend + ALS → Integration

This notebook implements `recommend()` — the single entry point that orchestrates all upstream modules  
and routes between the **personalised** path (returning customers) and the **popular** path (cold-start).

## 1. Imports & Artefact Loading

Four pkl artefacts are required at inference time:
- `similarity_matrix.pkl` — 3,665 × 3,665 ALS item-item cosine similarity (used by `recommend_products`)
- `product_catalogue.pkl` — StockCode, Description, category, avg_price, popularity_rank
- `model_context.pkl` — RF context model (3 features, works for all users incl. cold-start)
- `model_history.pkl` — RF history model (15 features, returning users only)

> **Encoding constants** (`segment_map`, `price_map`, `all_fav_cats`) must exactly match what  
> was used during training in `ml_model.ipynb`. Do not modify these values.

In [1]:
import pickle
import numpy as np
import pandas as pd
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# ── path setup: works whether run from /notebooks or repo root ────────────────
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ── shared modules ────────────────────────────────────────────────────────────
from src.constants    import PRODUCT_CATEGORIES, SAMPLE_PROFILES, build_user_profile
from src.rules_engine import apply_rules
from src.search_module import find_reachable_categories, find_popular_categories

# ── artefact loading ──────────────────────────────────────────────────────────
print('Loading artefacts...')
item_sim_df       = pickle.load(open('models/similarity_matrix.pkl',  'rb'))
product_catalogue = pickle.load(open('models/product_catalogue.pkl',  'rb'))
model_context     = pickle.load(open('models/model_context.pkl',      'rb'))
model_history     = pickle.load(open('models/model_history.pkl',      'rb'))
print('All artefacts loaded.')
print(f'  similarity_matrix : {item_sim_df.shape}')
print(f'  product_catalogue : {product_catalogue.shape}')
print(f'  model_context     : MultiOutputClassifier ({len(model_context.estimators_)} estimators, 3 features)')
print(f'  model_history     : MultiOutputClassifier ({len(model_history.estimators_)} estimators, 15 features)')

# ── encoding constants (MUST match ml_model.ipynb training) ──────────────────
# Estimators in both models are ordered by PRODUCT_CATEGORIES (the label column order used during training).
# all_fav_cats must match the one-hot encoding used to build the 15-feature history vector.
segment_map  = {'New': 0, 'Occasional': 1, 'Frequent': 2}
price_map    = {'Low': 0, 'Mid-Low': 1, 'Mid-High': 2, 'High': 3}
all_fav_cats = PRODUCT_CATEGORIES + ['Unknown']   # 9 items: 8 categories + Unknown fallback

Loading artefacts...


All artefacts loaded.
  similarity_matrix : (3665, 3665)
  product_catalogue : (3897, 5)
  model_context     : MultiOutputClassifier (8 estimators, 3 features)
  model_history     : MultiOutputClassifier (8 estimators, 15 features)


## 2. Inference Functions (from ml_model.ipynb Section 9)

These functions are defined and trained by **Member 3**. They are copied here verbatim so this  
notebook can run independently without re-importing from the training notebook.

**Do not modify these functions** — the feature ordering in `extract_history_features()` and the  
encoding maps must exactly match what the models were trained on.

### How the confidence blend works

```
confidence = 1 - 1 / (1 + n_purchases)

n = 0  → confidence = 0.00  → 100% context model (cold-start)
n = 1  → confidence = 0.50  → 50% context + 50% history
n = 5  → confidence = 0.83  → 17% context + 83% history
n = 10 → confidence = 0.91  → 9%  context + 91% history
n = 14 → confidence = 0.93  → 7%  context + 93% history
```

As a customer builds more history, the personalised history model dominates.

In [2]:
def extract_context_features(user_profile):
    """
    Build the 3-feature context vector used by Model A.
    Features: customer_segment (encoded), price_range (encoded), current month.
    Always available — works for new and returning customers alike.
    """
    return [[
        segment_map.get(user_profile['customer_segment'], 0),
        price_map.get(user_profile['price_range'], 0),
        pd.Timestamp.now().month
    ]]


def extract_history_features(user_profile):
    """
    Build the 15-feature history vector used by Model B.
    Features: n_purchases, n_categories, recency_days, avg_order_value,
              segment_enc, price_enc, favourite_category (one-hot × 9).
    Defaults to zero for all history fields when purchase_history is empty.
    """
    fav = user_profile.get('favourite_category') or 'Unknown'
    if fav not in all_fav_cats:
        fav = 'Unknown'
    fav_ohe = [1 if c == fav else 0 for c in all_fav_cats]   # 9-element one-hot
    return [[
        len(user_profile['purchase_history']),
        len(user_profile.get('purchased_categories', [])),
        user_profile.get('recency_days', 999),
        user_profile.get('avg_order_value', 0.0),
        segment_map.get(user_profile['customer_segment'], 0),
        price_map.get(user_profile['price_range'], 0),
        *fav_ohe
    ]]


def predict_product(user_profile, candidates=None):
    """
    Score candidate categories using a confidence-blended RF model.
    Works for ALL users — new users get 100% context model; returning users blend both.

    Parameters
    ----------
    user_profile : dict   from build_user_profile()
    candidates   : list   subset of PRODUCT_CATEGORIES to score (default: all 8)

    Returns
    -------
    list[tuple[str, float]]  [(category, probability), ...] sorted descending
    """
    cats       = candidates or PRODUCT_CATEGORIES
    n_hist     = len(user_profile['purchase_history'])
    confidence = 1.0 - 1.0 / (1.0 + n_hist)

    # Model A — context scores (always computed)
    X_ctx      = extract_context_features(user_profile)
    ctx_scores = np.array([
        est.predict_proba(X_ctx)[0][1] if len(est.classes_) > 1 else 0.0
        for est in model_context.estimators_
    ])

    # Model B — history scores (only meaningful for returning users)
    if n_hist > 0:
        X_hist      = extract_history_features(user_profile)
        hist_scores = np.array([
            est.predict_proba(X_hist)[0][1] if len(est.classes_) > 1 else 0.0
            for est in model_history.estimators_
        ])
    else:
        hist_scores = np.zeros(len(PRODUCT_CATEGORIES))

    # Blend by confidence — estimators are in PRODUCT_CATEGORIES order
    blended = (1.0 - confidence) * ctx_scores + confidence * hist_scores
    scores  = {cat: float(blended[i])
               for i, cat in enumerate(PRODUCT_CATEGORIES) if cat in cats}
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def recommend_products(user_profile, category, top_n=3):
    """
    Recommend specific products within a category using ALS item similarity.
    Finds products most similar to the customer's purchase history.
    Falls back to global popularity if the customer has no purchase history.

    Parameters
    ----------
    user_profile : dict   from build_user_profile()
    category     : str    one of PRODUCT_CATEGORIES
    top_n        : int    number of products to return (default 3)

    Returns
    -------
    list[dict]  [{'category', 'product', 'price', 'score'}, ...]
                score is cosine similarity float, or 0.0 if using popularity fallback
    """
    bought       = user_profile['purchase_history']
    valid_bought = [sc for sc in bought if sc in item_sim_df.columns]

    cat_items = product_catalogue[product_catalogue['category'] == category].copy()
    unowned   = cat_items[~cat_items['StockCode'].isin(bought)]

    # Fallback: no purchase history or no unowned items in category → popularity
    if not valid_bought or unowned.empty:
        top = unowned.nlargest(top_n, 'popularity_rank')
        return [{'category': category,
                 'product':  row['Description'],
                 'price':    round(float(row['avg_price']), 2),
                 'score':    0.0}
                for _, row in top.iterrows()]

    # ALS similarity: mean similarity of candidate products to all owned products
    valid_unowned = unowned[unowned['StockCode'].isin(item_sim_df.index)].copy()
    if valid_unowned.empty:
        top = unowned.nlargest(top_n, 'popularity_rank')
        return [{'category': category,
                 'product':  row['Description'],
                 'price':    round(float(row['avg_price']), 2),
                 'score':    0.0}
                for _, row in top.iterrows()]

    valid_unowned['score'] = (
        item_sim_df.loc[valid_unowned['StockCode'], valid_bought]
                   .mean(axis=1).values
    )
    top = valid_unowned.nlargest(top_n, 'score')
    return [{'category': category,
             'product':  row['Description'],
             'price':    round(float(row['avg_price']), 2),
             'score':    round(float(row['score']), 4)}
            for _, row in top.iterrows()]


print('Inference functions ready: predict_product(), recommend_products()')

Inference functions ready: predict_product(), recommend_products()


## 3. `recommend()` — Main Orchestrator

This is the single public-facing function for the entire system. It routes between two paths:

```
purchase_history = []          → COLD-START path
  apply_rules → find_popular_categories → get_popular_products
  recommendation_type = 'popular', scores = int buyer counts

purchase_history = [...]       → PERSONALISED path
  apply_rules → find_reachable_categories → predict_product → recommend_products
  recommendation_type = 'personalised', scores = float RF probabilities
```

### Output schema
```python
{
    'recommendation_type':  'personalised' | 'popular',
    'top_3_categories':     [(category, score), ...]  # exactly 3 items
    'recommended_products': [{category, product, price, score}, ...]  # exactly 9 items
    'eligible':             [category, ...]  # from apply_rules()
}
```

In [3]:
def recommend(user_profile: dict) -> dict:
    """
    Main recommendation entry point. Routes between personalised and cold-start paths.

    Parameters
    ----------
    user_profile : dict   built by build_user_profile() from src.constants

    Returns
    -------
    dict with keys:
        recommendation_type  : 'personalised' | 'popular'
        top_3_categories     : [(category, score), ...]   — 3 items
        recommended_products : [{category, product, price, score}, ...] — 9 items
        eligible             : [category, ...]            — from apply_rules()
    """
    # Step 1: Rules engine — gate eligible categories
    eligible = apply_rules(user_profile)

    # ── COLD-START PATH ───────────────────────────────────────────────────────
    # No purchase history → ML model and BFS cannot personalise.
    # Fall back to globally popular categories and products within the eligible set.
    if not user_profile['purchase_history']:
        categories = find_popular_categories(
            eligible, user_profile['price_range'], top_n=3
        )
        products = []
        for cat, _ in categories:
            products += get_popular_products(cat, top_n=3)

        return {
            'recommendation_type':  'popular',
            'top_3_categories':     categories,
            'recommended_products': products,
            'eligible':             eligible,
        }

    # ── PERSONALISED PATH ─────────────────────────────────────────────────────
    # Step 2: BFS search — narrow eligible to categories reachable from favourite
    shortlist = find_reachable_categories(user_profile, eligible)

    # Step 3: RF blend — score and rank the shortlisted categories
    categories = predict_product(user_profile, candidates=shortlist)[:3]

    # Step 4: ALS item similarity — recommend 3 products per top category
    products = []
    for cat, _ in categories:
        products += recommend_products(user_profile, category=cat, top_n=3)

    return {
        'recommendation_type':  'personalised',
        'top_3_categories':     categories,
        'recommended_products': products,
        'eligible':             eligible,
    }


print('recommend() ready.')

recommend() ready.


## 4. `get_popular_products()` — Cold-Start Helper

Owned by Member 4. Used exclusively on the cold-start path to fetch the most globally  
purchased products within a given category.

Returns `score` as **int** (buyer count), not float — this signals to `display_recommendation()`  
that it should render scores as `X buyers` rather than probabilities.

In [4]:
def get_popular_products(category: str, top_n: int = 3) -> list:
    """
    Return the most globally purchased products in a given category.
    Used on the cold-start path when no purchase history is available.

    Parameters
    ----------
    category : str   one of PRODUCT_CATEGORIES
    top_n    : int   number of products to return (default 3)

    Returns
    -------
    list[dict]  [{'category', 'product', 'price', 'score'}, ...]
                score is int (unique buyer count) — NOT a float probability
    """
    cat_items = product_catalogue[product_catalogue['category'] == category]
    top       = cat_items.nlargest(top_n, 'popularity_rank')
    return [
        {
            'category': category,
            'product':  row['Description'],
            'price':    round(float(row['avg_price']), 2),
            'score':    int(row['popularity_rank']),  # int, not float
        }
        for _, row in top.iterrows()
    ]


print('get_popular_products() ready.')

get_popular_products() ready.


## 5. `display_recommendation()` — Output Formatter

Handles both score types cleanly:
- `float` → rendered as RF probability (e.g. `0.7381`)
- `int` → rendered as buyer count (e.g. `142,309 buyers`)

The `isinstance(score, float)` check is the discriminator — it works because  
`get_popular_products()` strictly returns Python `int`, not numpy int64.

In [5]:
def display_recommendation(result: dict, customer_name: str = '') -> None:
    """
    Pretty-print a recommend() output dict.
    Handles float (personalised) and int (popular) score types.

    Parameters
    ----------
    result        : dict   output of recommend()
    customer_name : str    optional label for display
    """
    is_personalised = result['recommendation_type'] == 'personalised'
    tag = 'Recommended for you' if is_personalised else 'Popular right now'

    print(f"\n{'═'*65}")
    if customer_name:
        print(f"  Customer   : {customer_name}")
    print(f"  Mode       : {result['recommendation_type'].upper()}  ({tag})")
    print(f"{'─'*65}")
    print(f"  Eligible   : {result['eligible']}")

    print(f"\n  Top 3 Categories:")
    for rank, (cat, score) in enumerate(result['top_3_categories'], 1):
        label = f"{score:.4f}" if isinstance(score, float) else f"{score:,} buyers"
        print(f"    {rank}. {cat:<30} {label}")

    print(f"\n  Recommended Products:")
    prev_cat = None
    for item in result['recommended_products']:
        if item['category'] != prev_cat:
            print(f"    ── {item['category']} ──")
            prev_cat = item['category']
        label = f"{item['score']:.4f}" if isinstance(item['score'], float) \
                else f"{item['score']:,} buyers"
        print(f"      {item['product'][:45]:<45} £{item['price']:>6.2f}  {label}")
    print(f"{'═'*65}")


print('display_recommendation() ready.')

display_recommendation() ready.


## 6. Demo — All 5 SAMPLE_PROFILES

Running the full pipeline on all five sample profiles defined in `src/constants.py`.  
These use real CustomerIDs from the UCI Online Retail dataset.

| Profile | Segment | Spend | Favourite | Expected path |
|---|---|---|---|---|
| `gift_buyer` | Occasional | Low | Seasonal & Gifts | personalised |
| `home_decorator` | Frequent | Low | Home Decor | personalised |
| `kitchen_enthusiast` | Frequent | Mid-Low | Kitchen & Dining | personalised |
| `craft_lover` | Occasional | Low | Stationery & Craft | personalised |
| `new_customer` | New | Low | None | **popular** |

In [6]:
results = {}
for name, profile in SAMPLE_PROFILES.items():
    results[name] = recommend(profile)
    display_recommendation(results[name], customer_name=name)


═════════════════════════════════════════════════════════════════
  Customer   : gift_buyer
  Mode       : PERSONALISED  (Recommended for you)
─────────────────────────────────────────────────────────────────
  Eligible   : ['Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts', 'Toys & Games', 'Stationery & Craft', 'Fashion & Accessories', 'Garden & Outdoor', 'Food & Confectionery']

  Top 3 Categories:
    1. Home Decor                     0.6256
    2. Seasonal & Gifts               0.5409
    3. Stationery & Craft             0.2325

  Recommended Products:
    ── Home Decor ──
      PINK BABY BUNTING                             £  2.97  0.5555
      PARTY BUNTING                                 £  4.88  0.5206
      VINTAGE UNION JACK BUNTING                    £  8.51  0.4651
    ── Seasonal & Gifts ──
      WOODEN HAPPY BIRTHDAY GARLAND                 £  2.97  0.3700
      GARLAND WOODEN HAPPY EASTER                   £  1.25  0.2737
      VINTAGE CHRISTMAS STOCKING            


═════════════════════════════════════════════════════════════════
  Customer   : home_decorator
  Mode       : PERSONALISED  (Recommended for you)
─────────────────────────────────────────────────────────────────
  Eligible   : ['Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts', 'Toys & Games', 'Stationery & Craft', 'Fashion & Accessories', 'Garden & Outdoor', 'Food & Confectionery']

  Top 3 Categories:
    1. Home Decor                     0.8199
    2. Garden & Outdoor               0.3613
    3. Toys & Games                   0.2992

  Recommended Products:
    ── Home Decor ──
      AGED GLASS SILVER T-LIGHT HOLDER              £  0.64  0.4670
      VINTAGE GLASS T-LIGHT HOLDER                  £  0.86  0.4484
      SILVER HANGING T-LIGHT HOLDER                 £  1.63  0.4380
    ── Garden & Outdoor ──
      ANTIQUE GLASS DRESSING TABLE POT              £  2.98  0.3851
      ANTIQUE TALL SWIRLGLASS TRINKET POT           £  3.80  0.3460
      SMALL GLASS HEART TRINKET POT     


═════════════════════════════════════════════════════════════════
  Customer   : kitchen_enthusiast
  Mode       : PERSONALISED  (Recommended for you)
─────────────────────────────────────────────────────────────────
  Eligible   : ['Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts', 'Toys & Games', 'Stationery & Craft', 'Fashion & Accessories', 'Garden & Outdoor', 'Food & Confectionery']

  Top 3 Categories:
    1. Home Decor                     0.7181
    2. Kitchen & Dining               0.5447
    3. Garden & Outdoor               0.2577

  Recommended Products:
    ── Home Decor ──
      PARISIENNE SEWING BOX                         £ 12.28  0.3873
      REGENCY SUGAR TONGS                           £  2.43  0.3347
      PARISIENNE KEY CABINET                        £  5.71  0.3343
    ── Kitchen & Dining ──
      REGENCY CAKE SLICE                            £  4.92  0.3699
      REGENCY MILK JUG PINK                         £  3.20  0.3623
      REGENCY SUGAR BOWL GREEN      


═════════════════════════════════════════════════════════════════
  Customer   : craft_lover
  Mode       : PERSONALISED  (Recommended for you)
─────────────────────────────────────────────────────────────────
  Eligible   : ['Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts', 'Toys & Games', 'Stationery & Craft', 'Fashion & Accessories', 'Garden & Outdoor', 'Food & Confectionery']

  Top 3 Categories:
    1. Home Decor                     0.7381
    2. Stationery & Craft             0.5354
    3. Seasonal & Gifts               0.4517

  Recommended Products:
    ── Home Decor ──
      ROUND PURPLE CLOCK WITH SUCKER                £  0.19  0.4517
      FOLKART CLIP ON STARS                         £  0.43  0.4362
      JAZZ HEARTS ADDRESS BOOK                      £  0.25  0.4338
    ── Stationery & Craft ──
      MADRAS NOTEBOOK MEDIUM                        £  0.80  0.4523
      ASSORTED TUTTI FRUTTI PEN                     £  0.37  0.4517
      CARTOON  PENCIL SHARPENERS         

### Verification Checklist
Automated checks against the output schema requirements.

In [7]:
PASS = '\033[92m PASS \033[0m'
FAIL = '\033[91m FAIL \033[0m'

def check(label, condition):
    print(f'  [{PASS if condition else FAIL}] {label}')
    return condition

all_passed = True
print('\n── Schema & Routing Checks ──')
for name, result in results.items():
    profile = SAMPLE_PROFILES[name]
    print(f'\n{name}:')
    ok = True
    ok &= check('Has exactly 4 keys',
                set(result.keys()) == {'recommendation_type','top_3_categories','recommended_products','eligible'})
    ok &= check('recommendation_type is valid string',
                result['recommendation_type'] in ('personalised','popular'))
    ok &= check('Cold-start → popular',
                not (not profile['purchase_history'] and result['recommendation_type'] != 'popular'))
    ok &= check('Returning → personalised',
                not (profile['purchase_history'] and result['recommendation_type'] != 'personalised'))
    ok &= check('top_3_categories has exactly 3 items',
                len(result['top_3_categories']) == 3)
    ok &= check('recommended_products has exactly 9 items',
                len(result['recommended_products']) == 9)
    ok &= check('All product dicts have required keys',
                all({'category','product','price','score'} <= set(p.keys())
                    for p in result['recommended_products']))
    if result['recommendation_type'] == 'personalised':
        ok &= check('Personalised scores are float',
                    all(isinstance(p['score'], float) for p in result['recommended_products']))
    else:
        ok &= check('Popular scores are int',
                    all(type(p['score']) is int for p in result['recommended_products']))
    all_passed = all_passed and ok

print(f'\n{"All checks passed ✓" if all_passed else "Some checks FAILED — review above"}')


── Schema & Routing Checks ──

gift_buyer:
  [ PASS ] Has exactly 4 keys
  [ PASS ] recommendation_type is valid string
  [ PASS ] Cold-start → popular
  [ PASS ] Returning → personalised
  [ PASS ] top_3_categories has exactly 3 items
  [ PASS ] recommended_products has exactly 9 items
  [ PASS ] All product dicts have required keys
  [ PASS ] Personalised scores are float

home_decorator:
  [ PASS ] Has exactly 4 keys
  [ PASS ] recommendation_type is valid string
  [ PASS ] Cold-start → popular
  [ PASS ] Returning → personalised
  [ PASS ] top_3_categories has exactly 3 items
  [ PASS ] recommended_products has exactly 9 items
  [ PASS ] All product dicts have required keys
  [ PASS ] Personalised scores are float

kitchen_enthusiast:
  [ PASS ] Has exactly 4 keys
  [ PASS ] recommendation_type is valid string
  [ PASS ] Cold-start → popular
  [ PASS ] Returning → personalised
  [ PASS ] top_3_categories has exactly 3 items
  [ PASS ] recommended_products has exactly 9 items
  [ P

### Pipeline Trace — How a Recommendation is Built

Step-by-step trace for `home_decorator` to make the pipeline's reasoning visible.

In [8]:
profile = SAMPLE_PROFILES['home_decorator']
print('── Pipeline Trace: home_decorator ──')
print(f'  purchase_history   : {profile["purchase_history"]}')
print(f'  favourite_category : {profile["favourite_category"]}')
print(f'  customer_segment   : {profile["customer_segment"]}')
print(f'  price_range        : {profile["price_range"]}')
print()

eligible  = apply_rules(profile)
print(f'Step 1 — apply_rules()         : {eligible}')

shortlist = find_reachable_categories(profile, eligible)
print(f'Step 2 — find_reachable_cats() : {shortlist}')
pruned    = [c for c in eligible if c not in shortlist]
if pruned:
    print(f'         (pruned by BFS)        : {pruned}')

ranked    = predict_product(profile, candidates=shortlist)
print(f'Step 3 — predict_product()     :')
for cat, score in ranked:
    marker = '  ← top 3' if (cat, score) in ranked[:3] else ''
    print(f'           {cat:<30} {score:.4f}{marker}')

top3 = ranked[:3]
print(f'Step 4 — recommend_products() × 3 categories:')
for cat, _ in top3:
    prods = recommend_products(profile, category=cat, top_n=3)
    print(f'  [{cat}]')
    for p in prods:
        print(f'    {p["product"][:45]:<45} £{p["price"]:>6.2f}  sim={p["score"]:.4f}')

── Pipeline Trace: home_decorator ──
  purchase_history   : ['22174', '22791', '84946', '85123A']
  favourite_category : Home Decor
  customer_segment   : Frequent
  price_range        : Low

Step 1 — apply_rules()         : ['Home Decor', 'Kitchen & Dining', 'Seasonal & Gifts', 'Toys & Games', 'Stationery & Craft', 'Fashion & Accessories', 'Garden & Outdoor', 'Food & Confectionery']
Step 2 — find_reachable_cats() : ['Home Decor', 'Food & Confectionery', 'Stationery & Craft', 'Garden & Outdoor', 'Kitchen & Dining', 'Toys & Games', 'Seasonal & Gifts']
         (pruned by BFS)        : ['Fashion & Accessories']


Step 3 — predict_product()     :
           Home Decor                     0.8199  ← top 3
           Garden & Outdoor               0.3613  ← top 3
           Toys & Games                   0.2992  ← top 3
           Seasonal & Gifts               0.2537
           Kitchen & Dining               0.2510
           Stationery & Craft             0.2509
           Food & Confectionery           0.2071
Step 4 — recommend_products() × 3 categories:
  [Home Decor]
    AGED GLASS SILVER T-LIGHT HOLDER              £  0.64  sim=0.4670
    VINTAGE GLASS T-LIGHT HOLDER                  £  0.86  sim=0.4484
    SILVER HANGING T-LIGHT HOLDER                 £  1.63  sim=0.4380
  [Garden & Outdoor]
    ANTIQUE GLASS DRESSING TABLE POT              £  2.98  sim=0.3851
    ANTIQUE TALL SWIRLGLASS TRINKET POT           £  3.80  sim=0.3460
    SMALL GLASS HEART TRINKET POT                 £  2.09  sim=0.3239
  [Toys & Games]
    3D DOG PICTURE PLAYING CARDS                  £  2.95  sim=0.1593
    PLAY

## 7. Results & Discussion

### Two-Path Design

The `recommend()` function implements a **two-path recommendation pipeline**. The routing decision  
is based entirely on whether `purchase_history` is empty. This reflects a fundamental principle in  
recommender systems: a personalisation model has nothing to personalise when there is no signal.  
Rather than returning low-confidence or meaningless predictions, the system falls back to a globally  
optimal baseline — the most widely purchased products within the customer's eligible categories.

### Score Semantics

The two paths produce different score types, and this is intentional:

- **Personalised path** — scores are RF probabilities (0–1 floats). They represent the blended  
  probability that the customer will purchase from a given category, weighted by how much purchase  
  history the model has to work with. A score of 0.73 means the model assigns a 73% probability  
  to that category being purchased next.

- **Cold-start path** — scores are buyer counts (integers). They represent how many unique  
  customers in the historical dataset purchased at least one product in that category. A score  
  of 142,309 means 142,309 unique buyers across all transactions bought a Home Decor item.

The `recommendation_type` key signals which interpretation applies. `display_recommendation()`  
uses `isinstance(score, float)` to branch formatting accordingly.

### Observed Patterns

Home Decor consistently ranks highest on the personalised path for all returning customers,  
regardless of their favourite category. This reflects the category's dominance in the training  
data (142,309 buyer count, the highest of all 8 categories). The RF model learned a strong  
prior toward Home Decor based on how frequently it appeared in training labels. The customer's  
actual favourite category consistently appears second or third, showing that the history model  
is correctly capturing individual preference signal — it just cannot overcome the global prior  
of the most popular category.

The ALS product-level recommendations within each category are meaningfully personalised:  
`craft_lover` receives Stationery items with high cosine similarity to their 14 owned products,  
while `kitchen_enthusiast` receives Regency kitchenware matching the Regency items in their history.

### Limitations

1. **Home Decor dominance** — the RF models reflect the popularity skew in the training data.  
   A re-weighting step (inverse frequency weighting of category labels) could reduce this bias.

2. **Matrix sparsity** — at 1.54% density, many customer-product pairs have no signal for ALS  
   to learn from. The popularity fallback (`score=0.0`) in `recommend_products()` handles this,  
   but the recommendations for sparse customers are effectively popularity-ranked, not personalised.

3. **Cold-start constraint** — new customers are limited to 3 eligible categories (Rule 10).  
   This is conservative but correct for academic purposes; a production system might use  
   session context (device, referral, browsing) to broaden the eligible set.